# 05 — A/B Validation Design + App Wireframe

**目标**：把 04 的 raise-and-test candidates 转成可执行的实验计划，并草拟 Streamlit MVP 的页面结构。

**为什么是 05 而不是 cannibalization**：04 的 *headline finding* 是 98.5% cells 贴上界。这意味着 MVP 的瓶颈不是"再加一个协变量"，而是"把推荐转成可信的部署路径"。Cannibalization 留作 06 robustness。

**关键设计决定**：
- **Randomization unit**：`store` or `store-cluster`（DFF 是 store-week scanner data；线下零售价格通常是 store-level policy）。An online equivalent would randomize at user/session — flagged in `app_wireframe.md` as the SaaS-translation of this design.
- **Primary metric**：gross profit per store-week
- **Guardrails**：weekly units, weekly revenue, observed price compliance（complaint proxy unavailable in this dataset）
- **Estimator**：difference-in-means with store FE, cluster SE at store
- **MDE**：tested at 25% / 50% / 100% / 150% of model-predicted profit lift
- **Risk-tiered test design**: single-store flight / cluster RCT / standard A/B

**产出**：
1. `data/processed/experiment_candidates.csv` — 每个 top-10 candidate 的 test plan：current price, candidate price, σ, risk, recommended test type, sample size at 80% power × 50% MDE
2. `reports/ab_test_plan.md` — 实验设计 SOP，包含 randomization, metric, guardrails, sample size table, rollout sequencing
3. `reports/figures/ab_sample_size_curve.png` — n_storeweeks vs MDE 曲线
4. `docs/app_wireframe.md` — Streamlit 6 页 wireframe spec（Executive Summary / Demand Model / Counterfactual Simulator / Profit Optimizer / Experiment Design / Limitations）


In [ ]:
from __future__ import annotations
from pathlib import Path
from datetime import datetime
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED = PROJECT_ROOT / 'data' / 'processed'
REPORTS   = PROJECT_ROOT / 'reports'
FIGURES   = REPORTS / 'figures'
DOCS      = PROJECT_ROOT / 'docs'
for p in (PROCESSED, FIGURES, DOCS): p.mkdir(parents=True, exist_ok=True)

summary_lines: list[str] = []
def log(msg: str) -> None:
    print(msg); summary_lines.append(msg)

log('# A/B Validation Design — execution summary')
log(f'Generated: {datetime.now().isoformat(timespec="seconds")}')
log('')


## 1. Load top-10 candidates + cell baselines + modeling data


In [ ]:
top10 = pd.read_csv(PROCESSED / 'top_recommendations_diverse.csv')
cells_df = pd.read_parquet(PROCESSED / 'cell_baselines.parquet')
mdl = pd.read_parquet(PROCESSED / 'demand_modeling_dataset.parquet')

log('## 1. Load')
log(f'- top-10 portfolio candidates: {len(top10)}')
log(f'- cell_baselines rows: {len(cells_df):,}')
log(f'- modeling dataset rows: {len(mdl):,}')


## 2. Per-cell weekly profit variance σ

For each top-10 candidate cell `(brand, size_oz, store)`, compute the historical std of weekly gross profit `quantity × (unit_price − unit_cost)`. This σ is the variance input to the sample-size formula.


In [ ]:
mdl['weekly_profit'] = mdl['quantity'] * (mdl['unit_price'] - mdl['unit_cost'])

key_cols = ['brand_final', 'size_oz_rounded', 'STORE']
cell_var = (mdl.groupby(key_cols, observed=True)['weekly_profit']
              .agg(['mean', 'std', 'count'])
              .rename(columns={'mean':'profit_mean_wk','std':'profit_std_wk','count':'n_weeks'})
              .reset_index())

# Coerce STORE dtype to int64 for clean merge with top10 (which is int from CSV)
top10['STORE'] = top10['STORE'].astype('int64')
cell_var['STORE'] = cell_var['STORE'].astype('int64')

cand = top10.drop(columns=['n_weeks']).merge(cell_var, on=key_cols, how='left')
log('## 2. Weekly profit variance')
log(f'- σ (median across top-10): ${cand["profit_std_wk"].median():.2f}/store-week')
log(f'- σ (range): ${cand["profit_std_wk"].min():.2f} – ${cand["profit_std_wk"].max():.2f}')


## 3. Risk classification

Each candidate gets a `risk_flag ∈ {high, medium, low}` based on extrapolation severity, demand-shift magnitude, and price-jump size.

| Trigger | Threshold |
|---|---|
| `flag_extrapolation` | `opt_hits_upper == True` |
| `flag_large_q_change` | `q_lift_ratio` outside `[0.5, 1.5]` |
| `flag_high_price_jump` | `(opt_price − mean_p) / mean_p > 0.30` |
| `flag_low_baseline_volume` | `baseline_q < 10` units/week |

Classification: `high` if ≥2 flags, `medium` if 1 flag, `low` if 0 flags.


In [ ]:
cand['flag_extrapolation']      = cand['opt_hits_upper'].astype(bool)
cand['flag_large_q_change']     = (cand['q_lift_ratio'] > 1.5) | (cand['q_lift_ratio'] < 0.5)
cand['flag_high_price_jump']    = (cand['opt_price'] - cand['mean_p']) / cand['mean_p'] > 0.30
cand['flag_low_baseline_volume']= cand['baseline_q'] < 10

flag_cols = ['flag_extrapolation','flag_large_q_change','flag_high_price_jump',
             'flag_low_baseline_volume']
cand['n_flags'] = cand[flag_cols].sum(axis=1).astype(int)
cand['risk_flag'] = pd.cut(cand['n_flags'], bins=[-0.5, 0.5, 1.5, 99],
                            labels=['low','medium','high'])

log('## 3. Risk classification')
for f in flag_cols:
    log(f'- {f}: {int(cand[f].sum())}/{len(cand)} candidates trip')
log(f'- risk distribution: {cand["risk_flag"].value_counts().to_dict()}')


## 4. Recommended test type

| risk_flag | recommended_test_type | rationale |
|---|---|---|
| **high**   | `single_store_flight` (1 treat + 1 matched control, 8 weeks) | 限制 downside；synthetic-control style attribution |
| **medium** | `cluster_rct` (5 treat + 5 control stores, 6 weeks) | balance speed vs noise |
| **low**    | `standard_ab` (10 treat + 10 control stores, 4 weeks) | full power, fast read |

`single_store_flight` 解释：先在 1 家 treatment store 试新价 8 周，对照 1 家 size/baseline-volume 最相近的 store；用 difference-in-differences vs pre-period。


In [ ]:
TEST_TYPE = {
    'high':   'single_store_flight',
    'medium': 'cluster_rct',
    'low':    'standard_ab',
}
TEST_DURATION_WEEKS = {'single_store_flight': 8, 'cluster_rct': 6, 'standard_ab': 4}
TEST_STORES_PER_ARM = {'single_store_flight': 1, 'cluster_rct': 5, 'standard_ab': 10}
cand['recommended_test_type']  = cand['risk_flag'].astype(str).map(TEST_TYPE)
cand['planned_duration_weeks'] = cand['recommended_test_type'].map(TEST_DURATION_WEEKS).astype(int)
cand['planned_stores_per_arm'] = cand['recommended_test_type'].map(TEST_STORES_PER_ARM).astype(int)

log('## 4. Test type assignment')
log('- ' + ', '.join(f'{k}: {v}' for k, v in cand['recommended_test_type'].value_counts().items()))


## 5. Sample size formula

Two-sample equal-variance t-test, store-week as the unit:
$$ n_\text{per arm} = \frac{2(z_{1-\alpha/2} + z_{1-\beta})^2 \sigma^2}{\delta^2} $$

- α = 0.05 (two-sided), z = 1.96
- power 0.80 → z = 0.84
- δ = effective MDE = (MDE_pct × baseline_profit_per_week)
- σ = historical weekly profit std for the cell

For each candidate, compute required `n_storeweeks` at MDE = 50% of model-predicted lift, power = 0.80.


In [ ]:
def n_per_arm(sigma, delta, alpha=0.05, power=0.80):
    z_a = norm.ppf(1 - alpha/2)
    z_b = norm.ppf(power)
    if delta <= 0 or sigma <= 0:
        return float('inf')
    return float(2 * (z_a + z_b)**2 * sigma**2 / delta**2)

# Assume MDE = 50% of model-predicted lift (conservative — design test to detect
# half the optimistic point estimate, which still beats most pricing experiments)
cand['mde_dollars_50pct'] = 0.50 * cand['profit_lift_abs']
cand['n_storeweeks_per_arm_at_50pct_MDE_80pct_power'] = cand.apply(
    lambda r: n_per_arm(r['profit_std_wk'], r['mde_dollars_50pct']), axis=1)
cand['n_weeks_per_store'] = (cand['n_storeweeks_per_arm_at_50pct_MDE_80pct_power']
                             / cand['planned_stores_per_arm']).round(1)

log('## 5. Sample size')
log(f'- median n_storeweeks/arm @ 50% MDE, 80% power: '
    f'{cand["n_storeweeks_per_arm_at_50pct_MDE_80pct_power"].median():.0f}')
log(f'- median weeks per treatment store: {cand["n_weeks_per_store"].median():.1f} '
    f'(vs planned duration {cand["planned_duration_weeks"].median():.0f} weeks)')

# Diagnostic: which candidates are *underpowered* given planned design?
cand['underpowered'] = cand['n_weeks_per_store'] > cand['planned_duration_weeks']
log(f'- candidates whose required weeks > planned duration: {int(cand["underpowered"].sum())}/{len(cand)}')


## 6. Sample size curves

Single figure: required `n_storeweeks` per arm vs MDE (% of baseline weekly profit), at three power levels {0.70, 0.80, 0.90}. Uses median σ across top-10 as the representative cell.


In [ ]:
sigma_repr = float(cand['profit_std_wk'].median())
baseline_repr = float(cand['baseline_profit'].median())
mde_pct_grid = np.linspace(10, 200, 39)  # 10% – 200% of baseline weekly profit

fig, ax = plt.subplots(figsize=(7.5, 4.5))
for power in [0.70, 0.80, 0.90]:
    n_grid = [n_per_arm(sigma_repr, baseline_repr * pct/100, power=power)
              for pct in mde_pct_grid]
    ax.plot(mde_pct_grid, n_grid, label=f'power = {power:.2f}')
ax.set_xlabel('MDE (% of baseline weekly profit)')
ax.set_ylabel('Required n_storeweeks per arm')
ax.set_yscale('log')
ax.set_title(f'Sample size curve (σ={sigma_repr:.0f}, baseline={baseline_repr:.0f} $/store-wk, α=0.05)')
ax.axhline(40, color='gray', linestyle='--', alpha=0.5)
ax.text(180, 42, 'n=40 (10 stores × 4 wks)', color='gray', fontsize=8, ha='right')
ax.grid(alpha=0.3, which='both')
ax.legend(loc='upper right')
fig.tight_layout()
fig_path = FIGURES / 'ab_sample_size_curve.png'
fig.savefig(fig_path, dpi=130)
plt.close(fig)

log('## 6. Sample size curves')
log(f'- saved: {fig_path.relative_to(PROJECT_ROOT)}')
log(f'- representative σ = ${sigma_repr:.0f}/store-wk, baseline = ${baseline_repr:.0f}/store-wk')


## 7. Build & save `experiment_candidates.csv`


In [ ]:
out_cols = [
    'brand_final', 'size_oz_rounded', 'STORE',
    'mean_p', 'opt_price', 'opt_promo',
    'baseline_q', 'baseline_profit', 'opt_q', 'opt_profit',
    'profit_lift_abs', 'profit_lift_pct',
    'profit_std_wk', 'n_weeks',
    'flag_extrapolation', 'flag_large_q_change', 'flag_high_price_jump',
    'flag_low_baseline_volume', 'n_flags', 'risk_flag',
    'recommended_test_type', 'planned_duration_weeks', 'planned_stores_per_arm',
    'mde_dollars_50pct', 'n_storeweeks_per_arm_at_50pct_MDE_80pct_power',
    'n_weeks_per_store', 'underpowered',
]
exp_view = cand.rename(columns={
    'mean_p':'current_price', 'opt_price':'candidate_price',
    'opt_promo':'promo_status',
})
out_cols2 = [c if c not in ('mean_p','opt_price','opt_promo')
             else {'mean_p':'current_price','opt_price':'candidate_price','opt_promo':'promo_status'}[c]
             for c in out_cols]
exp_view = exp_view[out_cols2].round(3)
exp_path = PROCESSED / 'experiment_candidates.csv'
exp_view.to_csv(exp_path, index=False)
log('## 7. Experiment candidates')
log(f'- saved: {exp_path.relative_to(PROJECT_ROOT)}')
log('')
log('| brand | size | store | curr → cand | promo | risk | test type | weeks×stores |')
log('|---|---|---|---|---|---|---|---|')
for _, r in exp_view.iterrows():
    log(f'| {r["brand_final"]} | {r["size_oz_rounded"]:.2f} | {int(r["STORE"])} | '
        f'{r["current_price"]:.2f} → {r["candidate_price"]:.2f} | '
        f'{int(r["promo_status"])} | {r["risk_flag"]} | '
        f'{r["recommended_test_type"]} | '
        f'{int(r["planned_duration_weeks"])}w × {int(r["planned_stores_per_arm"])}st |')


## 8. Write `reports/ab_test_plan.md`


In [ ]:
plan_lines = []
P = plan_lines.append
P('# A/B Test Plan — Pricing Decision Engine MVP')
P(f'Generated: {datetime.now().isoformat(timespec="seconds")}  ·  source: `notebooks/05_ab_testing_design.ipynb`')
P('')
P('## Purpose')
P('Validate the raise-and-test candidates produced by `notebooks/04_counterfactual.ipynb`. '
  'The optimizer recommends candidate price increases for ~5,900 brand-size-store cells, '
  'but the same headline showed 98.5% bind at the upper guardrail. **No price change ships '
  'without controlled validation.**')
P('')
P('## Design summary')
P('| Element | Choice | Rationale |')
P('|---|---|---|')
P('| Randomization unit | store (or store-cluster) | DFF is store-week scanner data; offline retail '
  "prices are typically a store-level policy. *Online equivalent: user/session — see app wireframe.* |")
P('| Primary metric | gross profit per store-week | direct revenue−cost reading; matches optimizer objective |')
P('| Secondary metrics | units sold, revenue, observed price compliance | guardrails for stockout / margin erosion |')
P('| Estimator | DiD with store FE, cluster SE at store | accounts for store-level baseline differences |')
P('| α | 0.05 (two-sided) | standard |')
P('| Power | 0.80 | standard MVP target; 0.90 used for high-stakes price increases |')
P('| MDE | 50% of model-predicted profit lift | half the optimistic point estimate; still beats no-test status quo |')
P('')
P('## Risk-tiered test types')
P('| Risk tier | Test type | Stores per arm | Duration | Use when |')
P('|---|---|---|---|---|')
P('| **high**   | `single_store_flight`  | 1 + 1 matched control | 8 weeks | extrapolation past historical price band, large Q swings |')
P('| **medium** | `cluster_rct`          | 5 vs 5                | 6 weeks | one risk flag, otherwise normal |')
P('| **low**    | `standard_ab`          | 10 vs 10              | 4 weeks | clean candidate, within historical price band |')
P('')
P('## Top-10 portfolio test plan')
P('| brand | size | store | current → candidate | risk | test type | required n_storeweeks/arm | weeks/store | ')
P('|---|---|---|---|---|---|---|---|')
for _, r in exp_view.iterrows():
    P(f'| {r["brand_final"]} | {r["size_oz_rounded"]:.2f} | {int(r["STORE"])} | '
      f'${r["current_price"]:.2f} → ${r["candidate_price"]:.2f} | '
      f'{r["risk_flag"]} | `{r["recommended_test_type"]}` | '
      f'{int(r["n_storeweeks_per_arm_at_50pct_MDE_80pct_power"])} | '
      f'{r["n_weeks_per_store"]:.1f} |')
P('')
P('## Sample size formula')
P('Two-sample equal-variance t-test:')
P('$$ n_{\\text{per arm}} = \\frac{2(z_{1-\\alpha/2} + z_{1-\\beta})^2 \\sigma^2}{\\delta^2} $$')
P('where σ = historical weekly profit std for the cell, δ = MDE in dollars/store-week.')
P('')
P('## Guardrails (stop-test triggers)')
P('1. Treatment-arm units sold drops by > 40% vs control over rolling 2-week window → pause test')
P('2. Treatment-arm gross revenue drops > 20% vs control → pause test (margin floor breach risk)')
P('3. Observed treatment-store price compliance < 80% → re-instruct stores; if persists, drop arm')
P('4. Customer complaint proxy unavailable in DFF — flagged as future ingestion target')
P('')
P('## Rollout sequencing')
P('Run tests in waves to avoid same-store interference:')
P('1. **Wave 1 (weeks 1–8)**: all `single_store_flight` candidates (high risk) — limits downside')
P('2. **Wave 2 (weeks 9–14)**: all `cluster_rct` candidates (medium risk)')
P('3. **Wave 3 (weeks 15–18)**: all `standard_ab` candidates (low risk) — should be empty for MVP top-10')
P('After each wave, re-estimate elasticity on the new data and refresh the candidate list.')
P('')
P('## What this test does NOT validate')
P('1. Cross-store substitution (customers driving to a control store)')
P('2. Cross-brand cannibalization (planned for `06_portfolio_pricing_extension.ipynb`)')
P('3. Long-horizon price-image effects (>6 months)')
P('4. Competitor reaction (rivals see your shelf price too)')

ab_path = REPORTS / 'ab_test_plan.md'
ab_path.write_text('\n'.join(plan_lines), encoding='utf-8')
log('## 8. AB test plan')
log(f'- saved: {ab_path.relative_to(PROJECT_ROOT)} ({len(plan_lines)} lines)')


## 9. Write `docs/app_wireframe.md` — Streamlit MVP spec

6 pages, each with: data inputs (which artifact), main charts, controls.


In [ ]:
wf = []
W = wf.append
W('# Pricing & Promotion Decision Engine — Streamlit MVP Wireframe')
W(f'Generated: {datetime.now().isoformat(timespec="seconds")}  ·  source: `notebooks/05_ab_testing_design.ipynb`')
W('')
W('## Stack')
W('- `streamlit ≥ 1.30`, `pandas`, `pyarrow`, `plotly`, `altair`')
W('- All data read from `data/processed/*.parquet` and `*.csv` — no live DB')
W('- Sidebar: project name, model version (FROZEN block from `demand_model_summary.md`), date stamp')
W('')
W('---')
W('')
W('## Page 1 — Executive Summary')
W('**Audience**: VP / decision approver.')
W('- KPI tiles: total eligible cells, total predicted weekly profit lift across portfolio, '
  '# of high-risk candidates, top-3 portfolio brand-size-store cards')
W('- One-line headline: "98.5% of recommendations are raise-and-test candidates; no price '
  'changes ship without controlled validation."')
W('- Link buttons → other pages')
W('- Inputs: `top_recommendations_diverse.csv`, `cell_baselines.parquet`, '
  '`reports/counterfactual_summary.md` (rendered)')
W('')
W('## Page 2 — Demand Model')
W('**Audience**: data scientist peer.')
W('- Show frozen coefficients table from `model_coefficients.csv`')
W('- Holdout fit panel (RMSE, median APE) — be explicit it is a no-IV baseline')
W('- Brand-level price elasticity bar chart (own + cross + promo)')
W('- Markdown rendering of `reports/demand_model_summary.md` FROZEN block')
W('- Inputs: `model_coefficients.csv`, `demand_model_summary.md`, `figures/demand_*.png`')
W('')
W('## Page 3 — Counterfactual Simulator')
W('**Audience**: pricing analyst exploring "what if".')
W('- Selectors: brand_final, size_oz_rounded, STORE → loads cell baseline')
W('- Slider: candidate price (bounded by `[0.85·p_min, 1.15·p_max]`, with margin floor 1.05·cost)')
W('- Toggle: promo on/off')
W('- Live plot: predicted Q vs price (curve), profit vs price (curve), with baseline marker')
W('- Sensitivity sidebar: β_own ∈ {-1.90, -1.73, -1.50}, β_cross ∈ {0, 0.5, 0.65}, '
  'promo θ ∈ {0.30, 0.43, 0.51} → re-render curves')
W('- Inputs: `cell_baselines.parquet`, `model_coefficients.csv`')
W('')
W('## Page 4 — Profit Optimizer')
W('**Audience**: pricing analyst building a candidate list.')
W('- Filter widget: brand, size range, store, current_promo, min n_weeks')
W('- Table: filtered slice of `all_recommendations.csv` with sortable Δprofit and risk columns')
W('- "Add to experiment basket" button → writes selected rows to a session basket')
W('- Bar chart: top-N current view by predicted Δprofit')
W('- Inputs: `all_recommendations.csv`, `cell_baselines.parquet`')
W('')
W('## Page 5 — Experiment Design')
W('**Audience**: same analyst, now planning the test.')
W('- Render `experiment_candidates.csv` as interactive table; per-row expand shows: σ, MDE, '
  'required n_storeweeks, recommended test type, planned duration, flagged risks')
W('- Sample size widget: input σ + δ + α + power → output n per arm + n_storeweeks total + '
  'reuse the §6 curve')
W('- Render `reports/ab_test_plan.md` as a tab')
W('- "Export experiment basket" button → CSV download with the test plan rows for ops handoff')
W('- Inputs: `experiment_candidates.csv`, `figures/ab_sample_size_curve.png`, `ab_test_plan.md`')
W('')
W('## Page 6 — Limitations')
W('**Audience**: anyone who is about to use the recommendations.')
W('- Render `reports/counterfactual_summary.md` "Limitations" section + 03 limitations list')
W('- Banner: "These are candidate actions for experimentation, not production deployment."')
W('- Roadmap callout: cannibalization (06), competitor response (07), causal IV (08)')
W('- Inputs: `counterfactual_summary.md`, `demand_model_summary.md`')
W('')
W('---')
W('')
W('## Online translation note (for SaaS pricing)')
W('In a SaaS / e-commerce setting, the randomization unit shifts from `store` to `user / session`. '
  'The same design pattern applies: define a primary economic metric, compute σ from historical user-week '
  'spend, and size the test the same way. The DFF MVP uses store-level rollout because it matches the '
  'underlying scanner-panel structure; replacing the unit and the σ source is the only change needed.')
W('')
W('## Build sequence')
W('1. `app/main.py` (sidebar + router)')
W('2. `app/pages/1_executive_summary.py`')
W('3. `app/pages/2_demand_model.py`')
W('4. `app/pages/3_counterfactual_simulator.py`  ← shared `predict_q` helper imported from `src/`')
W('5. `app/pages/4_profit_optimizer.py`')
W('6. `app/pages/5_experiment_design.py`')
W('7. `app/pages/6_limitations.py`')
W('8. Deploy: `streamlit run app/main.py` locally → free-tier Streamlit Cloud for portfolio link')

wf_path = DOCS / 'app_wireframe.md'
wf_path.write_text('\n'.join(wf), encoding='utf-8')
log('## 9. App wireframe')
log(f'- saved: {wf_path.relative_to(PROJECT_ROOT)} ({len(wf)} lines)')


## 10. Limitations + handoff


In [ ]:
log('## 10. Limitations & handoff')
log('1. **Sample size assumes i.i.d. store-weeks**. With store FE + clustering, effective n is smaller; '
    'the formula here is a planning estimate, not a final design. For deployment, use a panel power '
    'simulation that respects the within-store correlation structure.')
log('2. **σ estimated from observed period at observed prices** — variance under treatment may differ '
    'if price shifts demand into a different regime. Re-estimate σ after wave 1 before sizing waves 2+.')
log('3. **No matched-control selection algorithm** documented for `single_store_flight` — should be '
    'added in a follow-up: nearest-neighbor on (baseline_q, mean_p, baseline_profit, brand portfolio '
    'mix) within the same census region.')
log('4. **Stockout / inventory guardrail is heuristic** — flagged via 40% units drop, but DFF lacks '
    'true stockout indicators. Real deployment needs an inventory feed.')
log('5. **No multiple-testing correction** across the 10 candidates — when running 10 tests in '
    'parallel, expect ~0.5 false positives at α=0.05. For MVP that is acceptable; production should '
    'apply Benjamini-Hochberg or sequential testing.')
log('')
log('## Handoff to Streamlit build')
log(f'- App wireframe: `{(DOCS / "app_wireframe.md").relative_to(PROJECT_ROOT)}`')
log(f'- Inputs the app needs (all already produced):')
log('    · data/processed/all_recommendations.csv')
log('    · data/processed/top_recommendations_diverse.csv')
log('    · data/processed/cell_baselines.parquet')
log('    · data/processed/model_coefficients.csv')
log('    · data/processed/experiment_candidates.csv')
log('    · reports/demand_model_summary.md, counterfactual_summary.md, ab_test_plan.md')
log('    · reports/figures/*.png')

# Write summary
out_path = REPORTS / 'ab_validation_summary.md'
out_path.write_text('\n'.join(summary_lines), encoding='utf-8')
log(f'\n- summary saved: {out_path.relative_to(PROJECT_ROOT)}')
